# Setting Up

As usual, we'll first want to select our Conda environment in the top right, and then we can begin running the analysis one cell at a time using `Ctrl + Enter`.

In [ ]:
from pathlib import Path
from typing import Literal
from warnings import catch_warnings, simplefilter

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from umap import UMAP

---

In [ ]:
# Here we're once again using relative paths to specify where our data is located.
# The `..` notation indicates that we want to step up one directory level
METADATA_FILE = Path("../lesson_3/metadata.tsv")
DATA_BASE_PATH = Path("../lesson_3/diann")

# And once again, here are our human-friendly data labels
METADATA_COLUMNS = [
    "Protein ID",
    "Protein",
    "Gene",
    "Entry Name",
    "Protein Description",
    "Peptide Sequence",
    "Modified Sequence",
    "Charge",
    "Unique ID",
]

In [ ]:
# Here once again are our collection of helper functions for loading data
def _map_metadata(data: pl.DataFrame, metadata: pl.DataFrame) -> pl.DataFrame:
    """A helper function used to give the raw data columns more human friendly names."""
    annotation = {
        "Protein.Group": "Protein ID",
        "Protein.Ids": "Protein",
        "Genes": "Gene",
        "Protein.Names": "Entry Name",
        "First.Protein.Description": "Protein Description",
        "Stripped.Sequence": "Peptide Sequence",
        "Modified.Sequence": "Modified Sequence",
        "Precursor.Charge": "Charge",
        "Precursor.Id": "Unique ID",
    }
    for row in metadata.iter_rows(named=True):
        annotation[row["FILE"]] = row["CONDITION"]
    return data.rename(annotation)


def load_data(data_path: Path, metadata_path: Path) -> pl.DataFrame:
    """A convenience function to load in the raw ion data from our base data path and automatically apply convenient metadata mapping to the columns."""
    data = pl.read_csv(
        data_path.joinpath("20220120_rklaas_SUN-schizo-cortex-layer1-3+4-6_MBR.pr_matrix.tsv.zst"), separator="\t"
    )
    metadata = pl.read_csv(metadata_path, separator="\t")
    return _map_metadata(data, metadata=metadata)


def transform_dataframe(data: pl.DataFrame) -> pl.DataFrame:
    """Transforms an input ions dataframe into UMAP and linear regression friendly format by dropping extra columns and flipping the axes."""
    return (
        data.select(pl.col("Unique ID"), cs.contains("Intensity"))
        .transpose(include_header=True, header_name="Sample ID", column_names="Unique ID")
        .with_columns(
            pl.col("Sample ID").str.split("_").list.get(0).alias("Group"),
            # Here we use a technique called `regular expressions` to selectively replace part of our data
            # Regular expressions are incredibly powerful, and well worth learning for working with any kind of text-based data
            pl.col("Sample ID").str.split(" ").list.get(0).str.replace(r"_[0-9]+$", "").alias("Subgroup"),
            pl.col("Sample ID").str.split("_").list.get(1).alias("Layer"),
        )
        .select(pl.col("Group", "Subgroup", "Layer"), cs.by_dtype(pl.Float64))
        .fill_null(np.random.default_rng(42).normal(loc=1e4, scale=1e3))
    )

In [ ]:
data = load_data(DATA_BASE_PATH, METADATA_FILE)
display(data)

In [ ]:
# Here we're transposing our dataframe while also parsing out some useful information from the column names
# Additionally, we fill all missing values with a sensible low number, as generally UMAP doesn't tolerate missing values
transformed_data = transform_dataframe(data)
display(transformed_data)

---

# Visualizing Mass Spectrometry Data

## Exercise 1
First, we want to visualize how our data clusters in various ways. The block below does the following:
- Separates out the categories we're interested in visualizing our data with.
- Runs UMAP to compress our predictor variables.
- Plots UMAP 1 and UMAP 2, while coloring each point by its `Group`.

Once you run the block and observe the results, try to create additional plots that color the data points instead by `Subgroup` and `Layer`. Do any seem more predictive of location than the alternatives?

**BONUS QUESTION:** Why do we consider our ion intensities "predictor variables" in this case?

In [ ]:
with catch_warnings():
    simplefilter("ignore")
    labels = transformed_data.select(pl.col("Group", "Subgroup", "Layer"))
    predictors = transformed_data.select(cs.exclude("Group", "Subgroup", "Layer"))

    # Here we're performing our embedding, again setting a random seed to ensure reproducibility
    umap_embedding = UMAP(n_components=2, random_state=42).fit_transform(predictors)

    # And here we're appending our labels back on for visualization purposes
    umap_embedding = pl.concat(
        [labels, pl.DataFrame({"umap_1": umap_embedding[:, 0], "umap_2": umap_embedding[:, 1]})], how="horizontal"
    )

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect("equal", "box")
    sns.scatterplot(data=umap_embedding, x="umap_1", y="umap_2", hue="Group")
    plt.xlabel("UMAP 1", fontdict={"fontsize": 16})
    plt.ylabel("UMAP 2", fontdict={"fontsize": 16})
    plt.title("UMAP of Schizphrenia Dataset", fontdict={"fontsize": 18})
    plt.legend(title="Group")
    plt.tight_layout()
    plt.show()

---

# Linear Regression

## Exercise 2
We again start by splitting our predictors out from our labels.

The block then does the following:
- It splits our data into training and testing sets.
- It creates a pipeline that scales and centers our data, and then fits a linear regression to it.
- It predicts the label of our test set based on their values.
- Finally it calculates how accurate the model was.

Once you've run the block, try to do the following:
1. Repeat the process for each label type we're interested in predicting. Which label are we able to build the best classifier for? What does that tell us about confounding variables in our dataset?
2. Assess how well the performance of our classifier aligns with what we observed via UMAP.
3. **BONUS CHALLENGE**: Rewrite the block below to use a [for loop](https://wiki.python.org/python/archive/ForLoop.html) to automatically run on all three of our class variables, rather than needing to manually adjust the `class_to_predict` variable.

In [ ]:
# This is just a handy selector variable that you can use for the exercise without
# needing to find everywhere we use its value in the block
class_to_predict: Literal["Group", "Subgroup", "Layer"] = "Group"

labels = np.ravel(transformed_data.select(pl.col(class_to_predict).rank("dense")))
predictors = transformed_data.select(cs.exclude("Group", "Subgroup", "Layer"))

# Here we split our dataset into training and test sets to avoid biasing our model by only giving it data that it's seen before
# The stratify parameter makes sure that each set contains the same proportion of class labels
# E.G. Our test set is 50% control and 50% schizophrenia when using Group as the label
# this ensure we have good representation of the feature space of each class and thus can build an accurate model of both
x_train, x_test, y_train, y_test = train_test_split(predictors, labels, test_size=0.2, random_state=42, stratify=labels)

pipeline = make_pipeline(StandardScaler(), LinearRegression(n_jobs=-1))

regression_classifier = pipeline.fit(X=x_train, y=y_train)
predictions = np.round(regression_classifier.predict(X=x_test))

accuracy = np.sum(predictions == y_test) / y_test.shape * 100
print(f"\nOur classifier for `{class_to_predict}` is {accuracy[0]:.2f} % accurate.")

---

## Challenges
1. Split our transformed dataset into two parts, one containing only the entries from layer 1-3 of the prefrontal cortex, and the other containing only entries from layer 4-6. _Hint:_ You may want to look into polar's `filter` function.
2. Run UMAP on each subset, coloring the data points based on the "Group" column. Do the points more clearly cluster here than they did previously? Why do you think that is?
3. Create a new classifier for each subset, trying again to predict the "Group" column. How does the performance of the model compare to when we included both sets of layers? _Hint:_ Make sure you're still making training and testing datasets.
4. Change the `random_state` parameter of your `train_test_split()` functions to another value. What happens to the model's accuracy? Why do you think that might be? What could we do to mitigate this?